# Re-score the tuning search: 5-fold CV, in-domain only

`tune.ipynb`'s first pass selected its winner by mean(A1, A2, B1, B2) — the
four reported generalization scenarios — which is a worse leak than the SVM's
own tuning ever had (`svm-tuning.ipynb` never touched anything outside
`training-clean.npz`). This notebook re-scores the same 10 `(lr, batch_size)`
configs Optuna already drew, this time by 5-fold CV on the training pool only
(`run_cv.py`), mirroring the SVM's `StratifiedKFold(n_splits=5)` objective
exactly. No OOD dataset is read here.

Reuses the fold assignments `cnn-latest/run_kfold.py` already wrote
(`cnn-latest/results/splits/clean-fold{0..4}.csv`) — the same folds the
published baseline's own k-fold CV and the SVM's `kfold.ipynb` used.

Per-fold models are not persisted (matches the SVM protocol: fit, score,
discard). The re-selected winner's *already-trained* single-split checkpoint
(`weights/trial{NN}.keras`, from the first tuning pass) is what gets promoted —
this notebook only decides which one that is, and overwrites the earlier,
leaky promotion of `weights/clean-seed42.keras`.

## Config

In [1]:
from pathlib import Path
import subprocess
import pandas as pd
from IPython.display import display, Markdown

HERE = Path.cwd()
if HERE.name != "cnn-revised":
    HERE = Path("training/notebooks/cnn-revised").resolve()
TRAINING_ROOT = HERE.parents[1]
TRIALS_CSV = HERE / "results" / "tuning_trials.csv"
CV_SCORES_CSV = HERE / "results" / "cv_scores.csv"
FOLDS = list(range(5))

tuned = pd.read_csv(TRIALS_CSV)
TRIAL_IDS = sorted(tuned.trial.astype(int).tolist())

cfg = pd.DataFrame([
    {"key": "here", "value": str(HERE)},
    {"key": "configs (from tune.ipynb)", "value": f"{len(TRIAL_IDS)} trials: {TRIAL_IDS}"},
    {"key": "folds", "value": FOLDS},
    {"key": "jobs", "value": f"{len(TRIAL_IDS)} x {len(FOLDS)} = {len(TRIAL_IDS) * len(FOLDS)}"},
    {"key": "fold splits (reused)", "value": str(HERE.parent / "cnn-latest" / "results" / "splits" / "clean-fold{k}.csv")},
])
display(cfg)
assert all((HERE.parent / "cnn-latest" / "results" / "splits" / f"clean-fold{f}.csv").is_file() for f in FOLDS)

,key,value
0,here,/home/seya/code/chord-detection/training/noteb...
1,configs (from tune.ipynb),"10 trials: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]"
2,folds,"[0, 1, 2, 3, 4]"
3,jobs,10 x 5 = 50
4,fold splits (reused),/home/seya/code/chord-detection/training/noteb...


## Run all (trial, fold) jobs

Each is an isolated `run_cv.py` process. Rows already in `cv_scores.csv` are
skipped (resume-friendly).

In [2]:
jobs = [(t, f) for t in TRIAL_IDS for f in FOLDS]
done = pd.read_csv(CV_SCORES_CSV) if CV_SCORES_CSV.exists() else pd.DataFrame(columns=["trial", "fold"])
pending = [
    (t, f) for t, f in jobs
    if done.empty or not ((done.trial == t) & (done.fold == f)).any()
]
display(Markdown(f"Pending **{len(pending)}** / {len(jobs)}"))

for trial, fold in pending:
    display(Markdown(f"### trial {trial} fold {fold}"))
    result = subprocess.run(
        ["uv", "run", "python", "run_cv.py", "--trial", str(trial), "--fold", str(fold)],
        cwd=HERE,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(f"trial{trial}/fold{fold} exited {result.returncode}")

display(Markdown("All jobs done." if not pending else "Ran all pending jobs."))

Pending **6** / 50

### trial 8 fold 4

I0000 00:00:1788403358.500318  150138 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


OK trial8/fold4: test_loss=0.000000 test_acc=1.0000


### trial 9 fold 0

...

OK trial9/fold3: test_loss=0.000026 test_acc=1.0000


### trial 9 fold 4

I0000 00:00:1788403917.329424  159222 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


OK trial9/fold4: test_loss=0.000000 test_acc=1.0000


Ran all pending jobs.

## CV scores per config

Mean over the 5 folds. Ranked by mean `test_loss` ascending (primary — in-domain
accuracy saturates near 1.0 for most configs and has little resolution, matching
what the SVM's own tuning saw), `test_accuracy` descending as tie-break.

In [3]:
cv = pd.read_csv(CV_SCORES_CSV)
assert len(cv) == len(TRIAL_IDS) * len(FOLDS), f"expected {len(TRIAL_IDS) * len(FOLDS)} rows, got {len(cv)}"

agg = cv.groupby("trial").agg(
    lr=("lr", "first"),
    batch_size=("batch_size", "first"),
    mean_test_loss=("test_loss", "mean"),
    std_test_loss=("test_loss", "std"),
    mean_test_accuracy=("test_accuracy", "mean"),
    std_test_accuracy=("test_accuracy", "std"),
    mean_val_loss=("val_loss", "mean"),
).reset_index()
agg = agg.sort_values(
    ["mean_test_loss", "mean_test_accuracy"], ascending=[True, False]
).reset_index(drop=True)

cv_winner_trial = int(agg.iloc[0].trial)
show = agg.copy()
show.insert(0, "", ["*" if t == cv_winner_trial else "" for t in show.trial])

display(Markdown(
    f"**CV winner: trial {cv_winner_trial}** — lr={agg.iloc[0].lr:.5g}, "
    f"batch_size={int(agg.iloc[0].batch_size)}, "
    f"mean_test_loss={agg.iloc[0].mean_test_loss:.6f} "
    f"(± {agg.iloc[0].std_test_loss:.6f}), "
    f"mean_test_accuracy={agg.iloc[0].mean_test_accuracy:.4f}. `*` marks it."
))
display(show.round(6))

**CV winner: trial 9** — lr=0.00028181, batch_size=32, mean_test_loss=0.000034 (± 0.000063), mean_test_accuracy=1.0000. `*` marks it.

,,trial,lr,batch_size,mean_test_loss,std_test_loss,mean_test_accuracy,std_test_accuracy,mean_val_loss
0,*,9,0.000282,32,0.000034,0.000063,1.000000,0.000000,0.000275
1,,4,0.000281,16,0.000039,0.000058,1.000000,0.000000,0.000389
2,,5,0.000801,64,0.000120,0.000234,1.000000,0.000000,0.001130
3,,8,0.000125,32,0.000161,0.000276,1.000000,0.000000,0.000272
4,,6,0.000472,16,0.000183,0.000358,0.999861,0.000312,0.000342
5,,0,0.000357,16,0.000205,0.000337,1.000000,0.000000,0.000734
6,,7,0.000750,32,0.000262,0.000395,0.999861,0.000312,0.000839
7,,2,0.000773,64,0.000376,0.000804,0.999861,0.000312,0.000471
8,,1,0.000170,64,0.000464,0.000973,0.999861,0.000312,0.000621
9,,3,0.001697,16,2.150571,1.961994,0.416593,0.532257,2.150151


## Re-promote

Overwrites the earlier (leaky, OOD-selected) `weights/clean-seed42.keras` with
the CV-selected winner's already-trained single-split checkpoint. Also reports,
for the record only, what the previous (leaky) selection had chosen and
whether it changed — this comparison is not part of the CV selection itself.

In [4]:
import shutil

src_w = HERE / "weights" / f"trial{cv_winner_trial:02d}.keras"
dst_w = HERE / "weights" / "clean-seed42.keras"
src_h = HERE / "results" / "history" / f"trial{cv_winner_trial:02d}.csv"
dst_h = HERE / "results" / "history" / "clean-seed42.csv"

assert src_w.is_file(), src_w
shutil.copy2(src_w, dst_w)
shutil.copy2(src_h, dst_h)

prev_leaky_winner = int(tuned.sort_values(
    ["mean_ood", "test_accuracy", "val_loss"], ascending=[False, False, True]
).iloc[0].trial)

display(Markdown(
    f"Promoted trial {cv_winner_trial} (CV-selected, in-domain only) → "
    f"`{dst_w.relative_to(TRAINING_ROOT)}`.\n\n"
    f"Previous (leaky, OOD-selected) pick was trial {prev_leaky_winner} — "
    + ("**same config**, no change." if prev_leaky_winner == cv_winner_trial
       else "**different config**, checkpoint replaced.")
))

Promoted trial 9 (CV-selected, in-domain only) → `notebooks/cnn-revised/weights/clean-seed42.keras`.

Previous (leaky, OOD-selected) pick was trial 0 — **different config**, checkpoint replaced.